In [ ]:
# ------------------ STEP 1: UPLOAD ------------------
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# ------------------ STEP 2: IMPORTS ------------------
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# ------------------ STEP 3: LOAD ------------------
df = pd.read_csv(file_name)

# ------------------ STEP 4: CLEAN ------------------
df = df.dropna()

# ------------------ STEP 5: FEATURE ENGINEERING ------------------

# Example (modify based on available columns)
if "age" in df.columns:
    df["age_group"] = pd.cut(df["age"], bins=[0,40,60,100], labels=[0,1,2])

if "cholesterol" in df.columns and "age" in df.columns:
    df["chol_age"] = df["cholesterol"] * df["age"]

if "trestbps" in df.columns and "age" in df.columns:
    df["bp_ratio"] = df["trestbps"] / (df["age"] + 1)

# ------------------ STEP 6: ENCODING ------------------
df = pd.get_dummies(df, drop_first=True)

if "heart_disease" not in df.columns:
    raise Exception("Target missing")

X = df.drop("heart_disease", axis=1)
y = df["heart_disease"]

# SAVE FEATURES
feature_names = X.columns.tolist()

# ------------------ STEP 7: REMOVE WEAK FEATURES ------------------
corr = X.corrwith(y).abs()
X = X[corr[corr > 0.03].index]   # relaxed threshold

print("Features after filtering:", X.shape[1])

# ------------------ STEP 8: SPLIT ------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ------------------ STEP 9: TRANSFORM ------------------
scaler = StandardScaler()
power = PowerTransformer()

X_train = scaler.fit_transform(X_train)
X_train = power.fit_transform(X_train)

X_test = scaler.transform(X_test)
X_test = power.transform(X_test)

# ------------------ STEP 10: SMOTE ------------------
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)

# ------------------ STEP 11: MODEL ------------------
model = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=10,
    min_child_weight=1,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    reg_lambda=5,
    reg_alpha=2,
    eval_metric='logloss'
)

# ------------------ STEP 12: CROSS VALIDATION ------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_score = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')

print(" CV Accuracy:", np.mean(cv_score))

# ------------------ STEP 13: TRAIN ------------------
model.fit(X_train, y_train)

# ------------------ STEP 14: TEST ------------------
pred = model.predict(X_test)
test_acc = accuracy_score(y_test, pred)

print(" FINAL TEST ACCURACY:", test_acc)

# ------------------ STEP 15: SAVE ------------------
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(power, open("power.pkl", "wb"))
pickle.dump(feature_names, open("features.pkl", "wb"))

files.download("model.pkl")
files.download("scaler.pkl")
files.download("power.pkl")
files.download("features.pkl")

Saving perfect_heart_dataset_4000.csv to perfect_heart_dataset_4000.csv
Features after filtering: 14
 CV Accuracy: 0.8426136363636363
 FINAL TEST ACCURACY: 0.8


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df["heart_disease"].value_counts())

heart_disease
0    2200
1    1800
Name: count, dtype: int64


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
import numpy as np

# ---------------- CONFIG ----------------
st.set_page_config(
    page_title="AI Heart Disease Predictor",
    page_icon="❤️",
    layout="wide"
)

# ---------------- LOAD ----------------
model = pickle.load(open("model.pkl", "rb"))
features = pickle.load(open("features.pkl", "rb"))

# OPTIONAL SCALER
try:
    scaler = pickle.load(open("scaler.pkl", "rb"))
except:
    scaler = None

# ---------------- HEADER ----------------
st.title("❤️ AI Heart Disease Predictor")
st.markdown("### Smart Health Risk Analysis System")

# ---------------- INPUT ----------------
col1, col2 = st.columns(2)

with col1:
    age = st.slider("Age", 20, 100, 50)
    sex = st.selectbox("Gender", ["Male", "Female"])
    sex = 1 if sex == "Male" else 0

    smoking = st.selectbox("Smoking", ["No", "Yes"])
    smoking = 1 if smoking == "Yes" else 0

    diabetes = st.selectbox("Diabetes", ["No", "Yes"])
    diabetes = 1 if diabetes == "Yes" else 0

    obesity = st.selectbox("Obesity", ["No", "Yes"])
    obesity = 1 if obesity == "Yes" else 0

with col2:
    cholesterol = st.slider("Cholesterol", 100, 350, 200)
    resting_bp = st.slider("Blood Pressure", 80, 200, 120)
    max_hr = st.slider("Max Heart Rate", 60, 220, 150)

    exercise = st.selectbox("Exercise Level", [0,1,2])
    stress = st.selectbox("Stress Level", [0,1,2])

    family = st.selectbox("Family History", ["No", "Yes"])
    family = 1 if family == "Yes" else 0

# ---------------- INPUT DATA ----------------
input_dict = {
    "age": age,
    "sex": sex,
    "cholesterol": cholesterol,
    "resting_bp": resting_bp,
    "max_heart_rate": max_hr,
    "smoking": smoking,
    "diabetes": diabetes,
    "obesity": obesity,
    "exercise_level": exercise,
    "stress_level": stress,
    "family_history": family
}

input_df = pd.DataFrame([input_dict])

# ---------------- MATCH FEATURES ----------------
for col in features:
    if col not in input_df.columns:
        input_df[col] = 0

input_df = input_df[features]

# ---------------- SCALE ----------------
if scaler is not None:
    input_df = scaler.transform(input_df)

# ---------------- PREDICT ----------------
if st.button("🔍 Predict Risk"):

    pred = model.predict(input_df)[0]

    try:
        prob = model.predict_proba(input_df)[0][1]
    except:
        prob = float(pred)

    if prob < 0.3:
        st.success(f"Low Risk ({prob:.2%})")
    elif prob < 0.7:
        st.warning(f"Medium Risk ({prob:.2%})")
    else:
        st.error(f"High Risk ({prob:.2%})")

    st.progress(int(prob * 100))

Writing app.py


In [ ]:
from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>